# Batch 2 CPU keyframes — SigLIP2 trên Kaggle T4 ×2
Notebook chạy hai process, mỗi GPU một image encoder. Giữ JPEG và mapping CPU; FP16 inference, L2/đầu ra FP32. Mỗi GPU phải đạt cosine ≥ 0.999 so với 6 vector batch1 trước khi chạy.

**Thiết lập:** Notebook private, Accelerator **GPU T4 ×2**, Internet bật để cài thư viện/tải checkpoint pin revision. Add Input: dataset `control` và các dataset `images-*` được tạo bởi packer. Không công khai dữ liệu. Chỉ chuyển dữ liệu khi việc dùng cloud phù hợp quy định BTC.

Lần đầu dùng gói `kaggle-sample` (~3.000 ảnh). Sau đó tháo sample, gắn control/ảnh của gói full. Chỉ gắn **một catalog** tại một thời điểm. Không cần video gốc.

Để lưu output bền vững, dùng **Save Version → Save & Run All**. Notebook giới hạn công việc 9,5 giờ, dành thời gian kết thúc trước hạn phiên. File trong phiên interactive chưa được lưu thành version không được xem là checkpoint bền vững.

In [ ]:
import subprocess, sys
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "open-clip-torch==3.3.0"], check=True)


In [ ]:
from pathlib import Path
import json, shutil, torch
INPUT_ROOT = Path("/kaggle/input")
WORK = Path("/kaggle/working")
RUN_FULL = False           # Đổi True sau khi benchmark đạt; sample catalog bị chặn ở bước full.
BATCH_SIZE = 32           # Nếu CUDA OOM: dùng 16 rồi chạy lại; không đổi precision giữa các phiên.
HOURS = 9.5
# Gắn tất cả Notebook Outputs của những phiên trước bằng Add Input.
RESUME_ROOTS = sorted({str(p.parent) for p in INPUT_ROOT.rglob("run-summary.json")})
catalogs = list(INPUT_ROOT.rglob("catalog.json"))
assert len(catalogs) == 1, f"Cần đúng một catalog, tìm được: {catalogs}"
CATALOG = catalogs[0]
CATALOG_DATA = json.loads(CATALOG.read_text())
assert torch.cuda.device_count() == 2, "Chọn Accelerator GPU T4 ×2"
print([torch.cuda.get_device_name(i) for i in range(2)])
print("Catalog:", CATALOG, "Frames:", CATALOG_DATA["frames"], "Sample:", CATALOG_DATA["sample"])
print("Previous saved outputs:", RESUME_ROOTS)
RUNNER = Path("/tmp/siglip2_runner.py")
shutil.copy2(CATALOG.parent / "siglip2_runner.py", RUNNER)
def command(output, hours, max_shards=0, extra_resume=()):
    cmd = [sys.executable, "-u", str(RUNNER), "--catalog", str(CATALOG),
           "--input-root", str(INPUT_ROOT), "--output", str(output),
           "--gpus", "2", "--precision", "fp16", "--batch-size", str(BATCH_SIZE),
           "--hours", str(hours), "--allow-partial-input"]
    if max_shards:
        cmd += ["--max-shards", str(max_shards)]
    for root in [*RESUME_ROOTS, *extra_resume]:
        cmd += ["--resume-root", str(root)]
    return cmd


## Benchmark trên cả hai GPU
Chọn tối đa 4 shard phân bố trong dữ liệu đang gắn. Đo cả kiểm tra shard/đọc ảnh/preprocess/inference/ghi kết quả. Benchmark tạo kết quả hoàn chỉnh có thể dùng lại ở bước full. Nếu parity không đạt, notebook dừng; không bỏ qua kiểm tra này.

In [ ]:
BENCH = WORK / "benchmark"
subprocess.run(command(BENCH, hours=0.5, max_shards=4), check=True)
summary = json.loads((BENCH / "run-summary.json").read_text())
gpu = [json.loads(p.read_text()) for p in BENCH.glob("gpu-*-summary.json")]
frames = sum(x["frames"] for x in gpu)
if frames:
    rate = frames / summary["wall_seconds"]
    print(f"Measured end-to-end: {frames} frames, {rate:.2f} images/s (including setup).")
    print(f"Catalog extrapolation: {CATALOG_DATA['frames']/rate/3600:.2f} h; excludes upload, not a guarantee.")
else:
    print("Benchmark reused prior outputs; no fresh throughput estimate.")
print("GPU summaries:", gpu)


## Chạy các shard còn lại
Chỉ bật RUN_FULL khi dùng catalog full. Có thể gắn từng nhóm dataset ảnh theo dung lượng tài khoản. `remaining_shards` nêu rõ phần chưa hoàn thành, kể cả nhóm ảnh chưa gắn.

Khi chuyển phiên: lưu phiên hiện tại thành version thành công; Add Input tất cả output cũ, giữ nguyên control/model/precision/thư viện. Output cũ được kiểm tra rồi bỏ qua, không phải copy lại vào `/kaggle/working`. Khi tải về để ghép cần lấy **tất cả các phiên**, gồm folder benchmark.

File `.run.lock` chặn hai process cha ghi chung thư mục. Nếu phiên bị kill và còn lock, chỉ xóa lock sau khi chắc chắn process cũ đã dừng. `.npz.tmp` không được coi là kết quả hoàn tất.

In [ ]:
if RUN_FULL:
    assert not CATALOG_DATA["sample"], "Tháo gói sample và gắn catalog full trước."
    subprocess.run(command(WORK / "embeddings", hours=HOURS, extra_resume=[str(BENCH)]), check=True)
    print((WORK / "embeddings/run-summary.json").read_text())
else:
    print("Benchmark-only. Review parity/speed, then set RUN_FULL=True with the full catalog.")


## Tải kết quả về
Sau khi version hoàn tất, tải toàn bộ Output. Dùng `scripts/merge_kaggle_batch2.py` tại máy local với catalog full và tất cả thư mục kết quả. Merger kiểm tra đủ shard, checksum mapping, cấu hình, norm, ID và đường dẫn local trước khi tạo index batch2 riêng. Không tự ghép batch1 hoặc đổi app.

Thông số T4 được đo trong notebook này. Kết quả GTX 1070 local không được ghi nhận thành benchmark T4.